# Loan Dataset: Ensemble Classification and Regression Modelling

This notebook contains two parts:

- Part A: Ensemble classification for Loan Approval Status
- Part B: Decision Tree regression for Maximum Loan Amount

All outputs are formatted clearly for coursework screenshots.

## 1. Import Required Libraries
Import all packages needed for preprocessing, modelling, evaluation, and plotting.

In [ ]:
# Import warnings to suppress non-critical warnings in outputs.
import warnings
# Ignore warnings to keep outputs clean for screenshots.
warnings.filterwarnings("ignore")
# Import numpy for numerical operations.
import numpy as np
# Import pandas for tabular data handling.
import pandas as pd
# Import matplotlib for plotting figures.
import matplotlib.pyplot as plt
# Import seaborn for statistical visualizations.
import seaborn as sns
# Import LabelEncoder for encoding non-numeric class labels.
from sklearn.preprocessing import LabelEncoder
# Import train_test_split for dividing datasets into train and test sets.
from sklearn.model_selection import train_test_split
# Import Logistic Regression classifier.
from sklearn.linear_model import LogisticRegression
# Import KNN classifier.
from sklearn.neighbors import KNeighborsClassifier
# Import VotingClassifier for ensemble learning.
from sklearn.ensemble import VotingClassifier
# Import confusion matrix metric.
from sklearn.metrics import confusion_matrix
# Import classification report metric.
from sklearn.metrics import classification_report
# Import ROC curve metric function.
from sklearn.metrics import roc_curve
# Import AUC score metric function.
from sklearn.metrics import roc_auc_score
# Import DecisionTreeRegressor for regression modelling.
from sklearn.tree import DecisionTreeRegressor
# Import plot_tree for visualizing decision trees.
from sklearn.tree import plot_tree
# Import MSE metric for regression evaluation.
from sklearn.metrics import mean_squared_error
# Import MAE metric for regression evaluation.
from sklearn.metrics import mean_absolute_error
# Import R2 metric for regression evaluation.
from sklearn.metrics import r2_score
# Set seaborn style for cleaner plots.
sns.set_style("whitegrid")

## 2. Load Or Reuse Preprocessed Datasets
This notebook reuses existing X_class, y_class, X_reg, and y_reg if available in memory.
If not available, it rebuilds them from the CSV file using the same preprocessing logic.

In [ ]:
# Check if all required preprocessed datasets already exist in memory.
datasets_available = ("X_class" in globals()) and ("y_class" in globals()) and ("X_reg" in globals()) and ("y_reg" in globals())
# Start fallback preprocessing if datasets are not already available.
if not datasets_available:
    # Print message indicating fallback preprocessing is running.
    print("Preprocessed datasets not found in memory. Rebuilding from CSV...")
    # Set the CSV file path.
    csv_path = "loan_approval_data.csv"
    # Load raw data from CSV into a DataFrame.
    df = pd.read_csv(csv_path)
    # Define helper function for normalized column-name matching.
    def normalize_name(text):
        # Convert to lowercase and remove spaces and underscores.
        return str(text).strip().lower().replace(" ", "").replace("_", "")
    # Initialize list for ID-like columns to drop.
    id_like_columns = []
    # Loop over all columns in the DataFrame.
    for col in df.columns:
        # Normalize current column name.
        normalized_col = normalize_name(col)
        # Check if column appears to be an identifier.
        if normalized_col == "id" or normalized_col.endswith("id") or normalized_col in {"loanid", "applicationid"}:
            # Add detected ID-like column to drop list.
            id_like_columns.append(col)
    # Drop ID-like columns from DataFrame.
    df = df.drop(columns=id_like_columns, errors="ignore")
    # Select numeric columns for mean imputation.
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    # Select categorical columns for mode imputation.
    categorical_columns = df.select_dtypes(exclude=[np.number]).columns.tolist()
    # Loop through numeric columns for missing-value imputation.
    for col in numeric_columns:
        # Compute mean value for current numeric column.
        mean_value = df[col].mean()
        # Fill missing numeric values with column mean.
        df[col] = df[col].fillna(mean_value)
    # Loop through categorical columns for missing-value imputation.
    for col in categorical_columns:
        # Compute mode value for current categorical column.
        mode_value = df[col].mode(dropna=True)[0]
        # Fill missing categorical values with column mode.
        df[col] = df[col].fillna(mode_value)
    # Create mapping from normalized names to original column names.
    normalized_to_original = {normalize_name(c): c for c in df.columns}
    # Define candidate names for classification target column.
    class_target_candidates = ["loan_approval_status", "Loan Approval Status", "loan_status", "approved"]
    # Define candidate names for regression target column.
    reg_target_candidates = ["max_allowed_loan", "Maximum Loan Amount", "maximum_loan_amount", "loan_amount_max"]
    # Initialize classification target placeholder.
    class_target_col = None
    # Loop through classification target candidates.
    for candidate in class_target_candidates:
        # Normalize candidate target name.
        candidate_key = normalize_name(candidate)
        # Check if normalized candidate exists in column mapping.
        if candidate_key in normalized_to_original:
            # Save matched original classification target column name.
            class_target_col = normalized_to_original[candidate_key]
            # Stop search after first match.
            break
    # Initialize regression target placeholder.
    reg_target_col = None
    # Loop through regression target candidates.
    for candidate in reg_target_candidates:
        # Normalize candidate regression target name.
        candidate_key = normalize_name(candidate)
        # Check if normalized candidate exists in column mapping.
        if candidate_key in normalized_to_original:
            # Save matched original regression target column name.
            reg_target_col = normalized_to_original[candidate_key]
            # Stop search after first match.
            break
    # Raise error if class target is not found.
    if class_target_col is None:
        # Stop execution due to missing class target column.
        raise ValueError("Could not find Loan Approval Status target column.")
    # Raise error if regression target is not found.
    if reg_target_col is None:
        # Stop execution due to missing regression target column.
        raise ValueError("Could not find Maximum Loan Amount target column.")
    # Build classification features by dropping class target.
    X_class = df.drop(columns=[class_target_col]).copy()
    # Build classification target from class target column.
    y_class = df[class_target_col].copy()
    # One-hot encode classification feature matrix.
    X_class = pd.get_dummies(X_class, drop_first=False)
    # Encode class target labels when target is non-numeric.
    if not pd.api.types.is_numeric_dtype(y_class):
        # Initialize label encoder for class target encoding.
        label_encoder = LabelEncoder()
        # Convert class labels into numeric labels.
        y_class = pd.Series(label_encoder.fit_transform(y_class), name=class_target_col)
    # Copy class target values to find approved rows for regression.
    approval_series = df[class_target_col].copy()
    # Set default approved value for numeric binary targets.
    approved_value = 1
    # Handle approved value detection for non-numeric class targets.
    if not pd.api.types.is_numeric_dtype(approval_series):
        # Convert class values to lowercase strings for matching.
        unique_values = [str(v).strip().lower() for v in approval_series.dropna().unique().tolist()]
        # Define common approval labels for matching.
        approval_candidates = ["approved", "approve", "yes", "y", "true", "1"]
        # Identify values that match known approval labels.
        matched_values = [v for v in unique_values if v in approval_candidates]
        # Update approved value if at least one match is found.
        if len(matched_values) > 0:
            # Select first matched approval label as approved value.
            approved_value = matched_values[0]
        # Convert approval series to normalized lowercase strings.
        approval_series = approval_series.astype(str).str.strip().str.lower()
    # Handle approved value detection for numeric class targets.
    if pd.api.types.is_numeric_dtype(df[class_target_col]):
        # Use 1 as approved value when present, else use maximum class value.
        approved_value = 1 if 1 in set(df[class_target_col].unique()) else df[class_target_col].max()
        # Keep numeric series unchanged for filtering.
        approval_series = df[class_target_col]
    # Filter only approved-loan rows for regression dataset.
    approved_df = df[approval_series == approved_value].copy()
    # Raise error if no approved records are available.
    if approved_df.shape[0] == 0:
        # Stop execution to avoid creating empty regression dataset.
        raise ValueError("No approved loans found for regression dataset.")
    # Build regression feature matrix by dropping regression and class targets.
    X_reg = approved_df.drop(columns=[reg_target_col, class_target_col], errors="ignore").copy()
    # Build regression target vector from regression target column.
    y_reg = approved_df[reg_target_col].copy()
    # One-hot encode regression feature matrix.
    X_reg = pd.get_dummies(X_reg, drop_first=False)
# Print message when existing datasets are reused from memory.
if datasets_available:
    # Confirm reuse of in-memory preprocessed datasets.
    print("Using existing preprocessed X_class, y_class, X_reg, and y_reg from memory.")
# Convert classification features to DataFrame with clean index.
X_class = pd.DataFrame(X_class).reset_index(drop=True)
# Convert classification target to Series with clean index.
y_class = pd.Series(y_class).reset_index(drop=True)
# Convert regression features to DataFrame with clean index.
X_reg = pd.DataFrame(X_reg).reset_index(drop=True)
# Convert regression target to Series with clean index.
y_reg = pd.Series(y_reg).reset_index(drop=True)
# Print final shape of classification features.
print("X_class shape:", X_class.shape)
# Print final shape of classification target.
print("y_class shape:", y_class.shape)
# Print final shape of regression features.
print("X_reg shape:", X_reg.shape)
# Print final shape of regression target.
print("y_reg shape:", y_reg.shape)

## Part A: Ensemble Classifier

This section builds two base learners, combines them with soft voting, evaluates all models, and compares ensemble performance to individual models.

### A1. Train-Test Split For Classification
Use an 80/20 split with stratification to preserve class balance.

In [ ]:
# Split classification dataset into train and test sets.
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    # Provide classification feature matrix.
    X_class,
    # Provide classification target vector.
    y_class,
    # Set test size to 20 percent.
    test_size=0.2,
    # Keep class distribution balanced in both splits.
    stratify=y_class,
    # Set random seed for reproducible split.
    random_state=42
)
# Print shape of classification training features.
print("Xc_train shape:", Xc_train.shape)
# Print shape of classification testing features.
print("Xc_test shape:", Xc_test.shape)
# Print shape of classification training target.
print("yc_train shape:", yc_train.shape)
# Print shape of classification testing target.
print("yc_test shape:", yc_test.shape)

### A2. Build Base Learners And Soft Voting Ensemble
Base learners used: Logistic Regression and KNN.
Then combine them using VotingClassifier with soft voting.

In [ ]:
# Create Logistic Regression base learner.
log_reg_base = LogisticRegression(max_iter=2000, random_state=42)
# Create KNN base learner with k equals 5.
knn_base = KNeighborsClassifier(n_neighbors=5)
# Create soft voting ensemble using the two base learners.
voting_ensemble = VotingClassifier(
    # Provide named estimators for ensemble construction.
    estimators=[("lr", LogisticRegression(max_iter=2000, random_state=42)), ("knn", KNeighborsClassifier(n_neighbors=5))],
    # Use soft voting to combine predicted probabilities.
    voting="soft"
)
# Build model dictionary for unified training and evaluation.
classification_models = {
    # Add Logistic Regression model entry.
    "Logistic Regression": log_reg_base,
    # Add KNN model entry.
    "KNN (k=5)": knn_base,
    # Add Voting Ensemble model entry.
    "Voting Ensemble (Soft)": voting_ensemble
}
# Train each classification model on the same training data.
for model_name, model_obj in classification_models.items():
    # Fit current model with classification training features and labels.
    model_obj.fit(Xc_train, yc_train)
    # Print confirmation that model training is complete.
    print(f"Trained model: {model_name}")

### A3. Evaluate Individual Models And Ensemble
For each model, display confusion matrix, classification report, ROC curve, and AUC.

In [ ]:
# Identify sorted unique class labels in test target.
class_labels = np.sort(np.unique(yc_test))
# Raise error if target is not binary for ROC/AUC workflow.
if len(class_labels) != 2:
    # Stop execution because this evaluation cell expects binary classification.
    raise ValueError("Binary target is required for this ROC/AUC setup.")
# Define positive class as higher numeric label.
positive_class = class_labels[-1]
# Initialize list to collect model comparison metrics.
ensemble_comparison_rows = []
# Create ROC plot figure for all models.
roc_fig, roc_ax = plt.subplots(figsize=(8, 6))
# Loop through every classification model for evaluation.
for model_name, model_obj in classification_models.items():
    # Print separator line for clear output sections.
    print("\n" + "=" * 90)
    # Print model heading for current evaluation block.
    print(f"Evaluation: {model_name}")
    # Print separator line again.
    print("=" * 90)
    # Predict class labels on classification test features.
    y_pred = model_obj.predict(Xc_test)
    # Predict positive class probabilities on test features.
    y_prob = model_obj.predict_proba(Xc_test)[:, 1]
    # Compute confusion matrix for current model.
    cm = confusion_matrix(yc_test, y_pred)
    # Print confusion matrix title for readability.
    print("Confusion Matrix:")
    # Display confusion matrix in tabular form.
    display(pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Pred 0", "Pred 1"]))
    # Plot confusion matrix heatmap for visual comparison.
    cm_fig, cm_ax = plt.subplots(figsize=(5, 4))
    # Draw heatmap with annotated counts.
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=cm_ax)
    # Set heatmap title with model name.
    cm_ax.set_title(f"Confusion Matrix - {model_name}")
    # Label x-axis as predicted label.
    cm_ax.set_xlabel("Predicted Label")
    # Label y-axis as true label.
    cm_ax.set_ylabel("True Label")
    # Apply tight layout for clean rendering.
    cm_fig.tight_layout()
    # Show confusion matrix heatmap.
    display(cm_fig)
    # Close confusion matrix figure after display.
    plt.close(cm_fig)
    # Compute classification report dictionary for metric extraction.
    report_dict = classification_report(yc_test, y_pred, output_dict=True, zero_division=0)
    # Print classification report title.
    print("Classification Report:")
    # Display classification report as rounded DataFrame.
    display(pd.DataFrame(report_dict).transpose().round(4))
    # Compute ROC coordinates for current model.
    fpr, tpr, _ = roc_curve(yc_test, y_prob, pos_label=positive_class)
    # Compute AUC value for current model.
    auc_value = roc_auc_score(yc_test, y_prob)
    # Plot current model ROC line onto shared ROC figure.
    roc_ax.plot(fpr, tpr, linewidth=2, label=f"{model_name} (AUC = {auc_value:.4f})")
    # Extract positive-class recall from report dictionary.
    positive_recall = report_dict[str(int(positive_class))]["recall"] if str(int(positive_class)) in report_dict else report_dict[str(positive_class)]["recall"]
    # Append model metrics for later comparison table.
    ensemble_comparison_rows.append({
        # Save model name in summary row.
        "Model": model_name,
        # Save positive-class recall in summary row.
        "Recall": positive_recall,
        # Save AUC score in summary row.
        "AUC": auc_value
    })
# Plot random baseline diagonal line on ROC figure.
roc_ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
# Set title for combined ROC comparison figure.
roc_ax.set_title("ROC Curve Comparison: Individual Models vs Ensemble")
# Set x-axis label for ROC comparison figure.
roc_ax.set_xlabel("False Positive Rate")
# Set y-axis label for ROC comparison figure.
roc_ax.set_ylabel("True Positive Rate")
# Add legend to ROC comparison figure.
roc_ax.legend(loc="lower right")
# Apply tight layout to ROC comparison figure.
roc_fig.tight_layout()
# Show ROC comparison figure.
display(roc_fig)
# Close ROC figure after display.
plt.close(roc_fig)

### A4. Compare Ensemble Performance With Individual Models
Sort and compare models by Recall and AUC to check whether the ensemble improves classification quality.

In [ ]:
# Convert ensemble comparison rows into a DataFrame.
ensemble_results_df = pd.DataFrame(ensemble_comparison_rows)
# Sort model comparison by Recall then AUC in descending order.
ensemble_results_df = ensemble_results_df.sort_values(by=["Recall", "AUC"], ascending=False).reset_index(drop=True)
# Print comparison heading for classification models.
print("Classification Model Comparison (Recall and AUC):")
# Display rounded model comparison table.
display(ensemble_results_df.round(4))

## Part B: Regression Modelling

This section trains and compares two Decision Tree Regressors on Maximum Loan Amount prediction.

### B1. Train-Test Split For Regression
Use an 80/20 split for regression dataset.

In [ ]:
# Split regression dataset into train and test sets.
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    # Provide regression feature matrix.
    X_reg,
    # Provide regression target vector.
    y_reg,
    # Set test size to 20 percent.
    test_size=0.2,
    # Set random seed for reproducibility.
    random_state=42
)
# Print shape of regression training features.
print("Xr_train shape:", Xr_train.shape)
# Print shape of regression testing features.
print("Xr_test shape:", Xr_test.shape)
# Print shape of regression training target.
print("yr_train shape:", yr_train.shape)
# Print shape of regression testing target.
print("yr_test shape:", yr_test.shape)

### B2. Build And Train Two Decision Tree Regressors
- DT1: Default fully grown tree
- DT2: Pruned tree with max_depth = 4

In [ ]:
# Create DT1 as fully grown default decision tree regressor.
dt1_model = DecisionTreeRegressor(random_state=42)
# Create DT2 as pruned decision tree regressor with max_depth equals 4.
dt2_model = DecisionTreeRegressor(max_depth=4, random_state=42)
# Train DT1 on regression training data.
dt1_model.fit(Xr_train, yr_train)
# Train DT2 on regression training data.
dt2_model.fit(Xr_train, yr_train)
# Print confirmation for model training completion.
print("Trained DT1 and DT2 models successfully.")

### B3. Evaluate DT1 And DT2
Evaluate both models using MSE, MAE, and R2 score, then compare results.

In [ ]:
# Predict regression targets on test data using DT1.
yr_pred_dt1 = dt1_model.predict(Xr_test)
# Predict regression targets on test data using DT2.
yr_pred_dt2 = dt2_model.predict(Xr_test)
# Compute MSE for DT1 predictions.
mse_dt1 = mean_squared_error(yr_test, yr_pred_dt1)
# Compute MAE for DT1 predictions.
mae_dt1 = mean_absolute_error(yr_test, yr_pred_dt1)
# Compute R2 score for DT1 predictions.
r2_dt1 = r2_score(yr_test, yr_pred_dt1)
# Compute MSE for DT2 predictions.
mse_dt2 = mean_squared_error(yr_test, yr_pred_dt2)
# Compute MAE for DT2 predictions.
mae_dt2 = mean_absolute_error(yr_test, yr_pred_dt2)
# Compute R2 score for DT2 predictions.
r2_dt2 = r2_score(yr_test, yr_pred_dt2)
# Create table comparing DT1 and DT2 regression metrics.
regression_comparison_df = pd.DataFrame({
    # Add model names column.
    "Model": ["DT1 (Default)", "DT2 (max_depth=4)"],
    # Add MSE metric column.
    "MSE": [mse_dt1, mse_dt2],
    # Add MAE metric column.
    "MAE": [mae_dt1, mae_dt2],
    # Add R2 metric column.
    "R2": [r2_dt1, r2_dt2]
})
# Print heading for regression model comparison.
print("Decision Tree Regression Comparison:")
# Display rounded regression comparison table.
display(regression_comparison_df.round(4))

### B4. Visualize DT1 And DT2 Trees
Plot both decision trees for interpretability.

For DT1, only the top levels are shown to keep the plot readable.

In [ ]:
# Create figure for DT1 visualization.
plt.figure(figsize=(24, 10))
# Plot top levels of DT1 to avoid an unreadably large plot.
plot_tree(
    # Provide DT1 model object.
    dt1_model,
    # Provide feature names for split labels.
    feature_names=Xr_train.columns,
    # Fill nodes with colors.
    filled=True,
    # Round node box corners.
    rounded=True,
    # Show only first three levels for clarity.
    max_depth=3,
    # Set font size for readability.
    fontsize=8
)
# Set title for DT1 tree plot.
plt.title("DT1 (Default Fully Grown) - Top 3 Levels")
# Apply tight layout to DT1 plot.
plt.tight_layout()
# Show DT1 tree plot.
plt.show()
# Create figure for DT2 visualization.
plt.figure(figsize=(24, 10))
# Plot DT2 tree which is already pruned by max depth.
plot_tree(
    # Provide DT2 model object.
    dt2_model,
    # Provide feature names for split labels.
    feature_names=Xr_train.columns,
    # Fill nodes with colors.
    filled=True,
    # Round node box corners.
    rounded=True,
    # Set font size for readability.
    fontsize=8
)
# Set title for DT2 tree plot.
plt.title("DT2 (Pruned) - max_depth = 4")
# Apply tight layout to DT2 plot.
plt.tight_layout()
# Show DT2 tree plot.
plt.show()

### B5. Predict Maximum Loan Amount For A New Sample Client
Create a sample client row based on an existing feature row and predict with both DT1 and DT2.

In [ ]:
# Copy the first training-row feature values as a valid sample template.
new_sample_client = Xr_train.iloc[[0]].copy()
# Update age value if age column exists in the feature set.
if "age" in new_sample_client.columns:
    # Set a sample age value for the new client.
    new_sample_client["age"] = 32
# Update income value if income column exists in the feature set.
if "income" in new_sample_client.columns:
    # Set a sample income value for the new client.
    new_sample_client["income"] = 78000
# Update loan amount value if loan_amount column exists in the feature set.
if "loan_amount" in new_sample_client.columns:
    # Set a sample requested loan amount for the new client.
    new_sample_client["loan_amount"] = 18000
# Print heading for new sample client preview.
print("New Sample Client Features:")
# Display the sample client feature row.
display(new_sample_client)
# Predict maximum loan amount using DT1 model.
predicted_loan_dt1 = dt1_model.predict(new_sample_client)[0]
# Predict maximum loan amount using DT2 model.
predicted_loan_dt2 = dt2_model.predict(new_sample_client)[0]
# Compute average prediction across both decision tree models.
predicted_loan_avg = (predicted_loan_dt1 + predicted_loan_dt2) / 2
# Print DT1 prediction in a formatted way.
print(f"Predicted Maximum Loan Amount by DT1: {predicted_loan_dt1:.2f}")
# Print DT2 prediction in a formatted way.
print(f"Predicted Maximum Loan Amount by DT2: {predicted_loan_dt2:.2f}")
# Print average of both model predictions.
print(f"Average of DT1 and DT2 Predictions: {predicted_loan_avg:.2f}")

## Final Summary
This notebook completed:

- Ensemble classification with soft voting and full evaluation
- Comparison of ensemble against individual base learners
- Regression modelling with two decision trees
- Metric comparison for DT1 and DT2
- Tree visualizations
- Prediction for a new sample client

All outputs are displayed clearly for coursework screenshots.